# Adım 3 — Metin çıkarma ve kontrol

1. hücre `data/legal/word/*.docx` dosyalarından `data/clean/*.txt` üretir.  
2. hücre çıktıyı kontrol eder. Her satırın sonunda `OK` olmalı.

In [ ]:
from chunky.extract import main

main()

In [ ]:
from pathlib import Path
import re

ROOT = Path.cwd().parent   # notebook "notebooks/" içinde çalışır, bir üstü repo kökü
CLEAN = ROOT / "data" / "clean"

EXPECTED_HEADERS = {
    "01_kanun_2918": 193, "02_trafik_yonetmeligi": 222, "03_meb_surucu_kursu": 74,
    "04_surucu_saglik": 14, "05_arac_muayene": 30, "06_elektrikli_skuter": 34,
}
HEADER = re.compile(r"^(?:Ek |Geçici |EK |GEÇİCİ )?(?:Madde|MADDE)\s+\d+(?:/[A-ZÇ])?\s*[–\-—]")
# Yapışık not işareti: harf veya ")" ardından 1-2 rakam ve boşluk/satır sonu (15x24 gibi ölçüler hariç)
GLUED = re.compile(r"(?<![0-9x])[a-zçğıöşü)…][0-9]{1,2}(?= |$)", re.M)

MUST_HAVE = {   # tablolar ve ekler metinde olmalı
    "01_kanun_2918": ["KARAYOLLARI TRAFİK KANUNU", "(1) Sayılı Liste"],
    "02_trafik_yonetmeligi": ["Otomobil (M1), (M1G), | 50 | 90 | 110 | 120", "1 SAYILI CETVEL"],
    "04_surucu_saglik": ["Turnike"],
}
MUST_NOT_HAVE = {  # notlar ve değişiklik tabloları metinde olmamalı
    "*": ["Yönetmeliğin Yayımlandığı", "Değiştiren Kanunun", "GÖSTERİR TABLO", "için tıklayınız", "____"],
    "02_trafik_yonetmeligi": ["Bu değişikliğin birinci fıkrası yayımı tarihinden 30 gün sonra"],
    "03_meb_surucu_kursu": ["Bu değişiklik yayımı tarihinden 6 ay sonra"],
    "04_surucu_saglik": ["Danıştay Sekizinci Dairesinin 14/6/2008"],
    "05_arac_muayene": ["7/1/2021 tarihli ve 31357 sayılı"],
}

all_ok = True
for name, expected in EXPECTED_HEADERS.items():
    text = (CLEAN / f"{name}.txt").read_text(encoding="utf-8")
    headers = sum(1 for line in text.splitlines() if HEADER.match(line))
    missing = [s for s in MUST_HAVE.get(name, []) if s not in text]
    leftover = [s for s in MUST_NOT_HAVE["*"] + MUST_NOT_HAVE.get(name, []) if s in text]
    glued = [m.group(0) for m in GLUED.finditer(text)]
    ok = headers == expected and not missing and not leftover and not glued
    all_ok &= ok
    print(f"{name:23} başlık {headers:3}/{expected:3}  eksik:{missing}  kalıntı:{leftover}  "
          f"yapışık rakam:{glued[:5]}  {'OK' if ok else 'KONTROL ET'}")
print("\nHEPSİ OK" if all_ok else "\nSORUN VAR")
